In [ ]:
import requests

In [ ]:
!pip install openai

In [6]:
from openai import OpenAI



class dpsk:
    def __init__(self, api_key, messages_limit=7, prompt="You are a helpful assistant", model="deepseek-chat"):
        self.client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
        self.messages_limit = messages_limit
        self.del_history(prompt)
        self.model = model

    def chat(self, text, stream=False, model=None):
        model = self.model if not model else model
        self.messages.append({"role": "user", "content": text})

        response = self.client.chat.completions.create(
            model=model,
            messages=self.messages,
            stream=stream)
        self.messages.append({
            "role": response.choices[0].message.role,
            "content": response.choices[0].message.content
        })

        while len(self.messages)>=self.messages_limit:
            for i, msg in enumerate(self.messages):
                if i>0 and msg["role"]!="system":
                    del self.messages[i]
                    break

        return response.choices[0].message.content

    def del_history(self, prompt="You are a helpful assistant"):
        if prompt:
            self.prompt = prompt
        self.messages = [{"role": "system", "content": self.prompt}]



#if __name__=="__main__":
#    ai = dpsk("your api key")
#    while True:
#        print(f"DeepSeek: {ai.chat(input("User: "))}\n")

In [24]:
!pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 77.2 MB/s eta 0:00:00


In [25]:
import re
from pymorphy3 import MorphAnalyzer

morph = MorphAnalyzer()

def preprocess(text):
    if not isinstance(text, str):
        return ""

    # Нижний регистр
    text = text.lower()

    # Замена ё -> е
    text = text.replace("ё", "е")

    # Удаление HTML
    text = re.sub(r"<.*?>", " ", text)

    # Удаление ссылок
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Оставляем только буквы и цифры
    text = re.sub(r"[^а-яa-z0-9\s]", " ", text)

    # Удаляем лишние пробелы
    text = re.sub(r"\s+", " ", text).strip()

    # Лемматизация
    lemmas = [
        morph.parse(word)[0].normal_form
        for word in text.split()
    ]

    return " ".join(lemmas)

In [54]:
def contains_red_flag(text, red_flag):
    pattern = rf"\b{re.escape(red_flag)}\w*"
    return bool(re.search(pattern, text, re.IGNORECASE))


def contains_red_flags(text, red_flags):
  preprocessed_text = preprocess(text)
  return any(contains_red_flag(preprocessed_text, flag) for flag in red_flags)


red_flags = [
    # Критические события
    "банкрот",
    "дефолт",
    "ликвид",
    "несостоятель",
    "конкурс",
    "наблюден",

    # Финансы
    "долг",
    "задолж",
    "неплатеж",
    "реструктур",
    "ликвидност",
    "убыт",
    "просроч",

    # Суды
    "иск",
    "суд",
    "арбитраж",
    "взыск",
    "оспар",

    # Правоохранительные органы
    "арест",
    "пристав",
    "конфиск",
    "уголов",
    "обвин",
    "задерж",
    "обыск",

    # Мошенничество
    "мошеннич",
    "хищен",
    "корруп",

    # Регуляторы
    "штраф",
    "санкц",
    "наруш",
    "провер",
    "предпис",
    "лиценз",
    "аннулир",

    # Строительство
    "замороз",
    "приостанов",
    "останов",
    "долгостро",
    "перенос",
    "срыв",

    # Кредиторы
    "кредитор",
    "требован",

    # Репутация
    "жалоб",
    "дольщик"
]

In [4]:
prompt = """Ты — модель классификации рисков.

Задача:
Оценить риск для компании по тексту новости.

Классы:
0 — новость не содержит негативной информации или является положительной.
1 — слабый негативный сигнал.
2 — существенный негативный сигнал.
3 — критический негативный сигнал (банкротство, ликвидация, дефолт, уголовное дело, арест активов, невозможность продолжать деятельность и т.п.).

Правила:
- Анализируй только текст новости.
- Если новость не относится к компании или информации недостаточно — возвращай 0.
- Не объясняй решение.
- Верни только одно число: 0, 1, 2 или 3."""

In [56]:
llm = dpsk("API_KEY", prompt=prompt, model="deepseek-chat")
examples = [
    # 0 — нет риска
    "Компания успешно завершила строительство жилого комплекса и досрочно ввела его в эксплуатацию.",

    # 0 — нейтральная новость
    "Совет директоров компании утвердил годовой отчет и объявил о проведении общего собрания акционеров.",

    # 1 — низкий риск
    "Компания сообщила о переносе сроков сдачи одного из жилых объектов на один месяц из-за неблагоприятных погодных условий.",

    # 2 — средний риск
    "Против компании подан иск на сумму 500 млн рублей. Руководство заявило, что намерено оспаривать требования в суде.",

    # 3 — высокий риск
    "Арбитражный суд признал компанию банкротом и ввел процедуру конкурсного производства."
]
for text in examples:
  if contains_red_flags(text, red_flags):
    print(llm.chat(text))

AuthenticationError: Error code: 401 - {'error': {'message': 'Authentication Fails, Your api key: ****_KEY is invalid', 'type': 'authentication_error', 'param': None, 'code': 'invalid_request_error'}}

In [12]:
import pandas as pd

df = pd.read_csv("df.csv")

In [27]:
def predict_risk(row):
    if contains_red_flags(row["content"], red_flags) or row["sentiment"] == "negative":
        return int(llm.chat(row["content"]))
    return 0

df["risk"] = df.apply(predict_risk, axis=1)

In [29]:
df['corr_risk'] = df['content'].apply(lambda text: int(llm.chat(text)))

In [47]:
for i in df[~((df['risk'] == df["corr_risk"]) | ((df['risk'] > 0) & (df['sentiment'] == 'negative')))]['title']:
  print(i)

ЖК «Турист» в Ростокино исключён из реестра продаж
Премиальный проект на Малой Грузинской передан новому девелоперу
Дольщики ЖК «Варшавские ворота» получили ключи спустя год задержки
Апарт-отель ORO в центре Москвы перешёл к новому владельцу
Ташир выкупил проблемный ЖК «Деко Резиденс» в Даниловском районе
В Химках инвестор меняет проект ЖК «Планерный квартал» из-за спроса
В Сочи проблемы с разрешением на ввод ЖК «Южное море-2»


In [55]:
print(sum(df['content'].apply(lambda text: contains_red_flags(text, red_flags))))
print(sum((df['content'].apply(lambda text: contains_red_flags(text, red_flags))) | (df['sentiment'] == 'negative')))

16
18


In [32]:
df

,id,project_id,project_name,developer,title,content,date,source,category,sentiment,created_at,risk,corr_risk
0,943c7fda728c4ce683e21e6dfd8511b0,2ca4d5c5067dea9fabd4bdbecc6d3cb1,Горизонт,Бизнеслэнд,Строительство ЖК «Горизонт» в Щёлково вновь за...,Правительство Московской области приостановило...,2025-07-10,Интерфакс,строительство,negative,2026-07-02 17:08:40.190799,2,2
1,6001d12a318a48c0b66f6df78fbebc12,c6bcf2246d22d85f1e0b6c7a965b58df,Турист,Алтай,ЖК «Турист» в Ростокино исключён из реестра пр...,Проект реконструкции гостиницы «Турист» в райо...,2025-08-20,Коммерсантъ,финансы,negative,2026-07-02 17:08:40.190799,0,1
2,28beee68ffe44f1d8f720a4b7404cb41,0dae2a614a344f51b839f16cec512bc7,Венеция,Восточная инвестиционно-строительная компания,Суд обязал застройщика «Венеции» в Старой Купа...,Арбитражный суд Московской области удовлетвори...,2025-09-05,Ведомости,судебные,neutral,2026-07-02 17:08:40.190799,1,1
3,3fa33f1626e945afa1350a758245c8af,568e7d17de308b40e4f140f192d2847d,Сибирячка,СТРОЙ-ПЛЮС,Новосибирский ЖК «Сибирячка» признан долгостроем,Объект на улице Сибиряков-Гвардейцев в Новосиб...,2026-01-15,ТАСС,строительство,negative,2026-07-02 17:08:40.190799,2,1
4,7831e934963548e6ac61c194c006f0fc,5fc5ccc257c198fdb83ea90cf8acd259,Мюр и Мерилиз / Дом на Малой Грузинской,October Group,Премиальный проект на Малой Грузинской передан...,October Group вышла из проекта реконструкции и...,2025-11-02,РБК,девелопмент,neutral,2026-07-02 17:08:40.190799,1,0
5,d35eb128e632419090d62609f87d5266,54a993d2868cf0d8fa56e8003b3189c9,Дом по ул. Хади Атласи,СТРОЙРЕСУРС ИНВЕСТ,В Казани задерживается выдача разрешения на вв...,Стройнадзор Татарстана отложил выдачу РВЭ для ...,2026-02-12,Реальное время,разрешения,negative,2026-07-02 17:08:40.190799,2,1
6,a64ccb11aca446cfae5564c8dc69c159,9affa3f88911b987487ed94f2727a1d5,Ягоды,КОРАЛ РЭД,ЖК «Ягоды» в Уссурийске получил новый график с...,Застройщик «КОРАЛ РЭД» объявил о переносе срок...,2025-12-20,Восток-Медиа,график,neutral,2026-07-02 17:08:40.190799,1,1
7,0e515b00b0f24f5b92b550ed0e39dcbb,87e62ebc38ad5c731e9924b1cb535e2c,Крепость,Инотек,Элитный проект «Крепость» в Красной Поляне ост...,Краснодарский краевой суд запретил строительст...,2025-10-30,Югополис,судебные,negative,2026-07-02 17:08:40.190799,2,2
8,4381b4d99fad45c08022a54ee0e975fc,ea383acbbdef6ff7806f8845b9450ace,Светлый мир БиоПолис,ССД МСК ВОСТОК,«БиоПолис» в Мещерино заморожен из-за смены ге...,Строительство инновационного ЖК «Светлый мир Б...,2026-01-25,Московский комсомолец,строительство,negative,2026-07-02 17:08:40.190799,2,3
9,0a9a4454ea9741138911656ac7fa1d90,9344275ab84a238f5c3ee04f966caa39,ASTRVM / Дом Билибина,ПРОМСТРОЙКОМПЛЕКТ,В Петроградском районе завершено строительство...,Заключение о соответствии получено для элитног...,2026-05-20,Санкт-Петербургские ведомости,ввод,positive,2026-07-02 17:08:40.190799,0,0
